# Custom CNN architecture

In [1]:
# Run the previous notebook to load all its classes and functions
%run cbis_ddsm_create_ROI_dataset_classification_rs1.ipynb

In [ ]:
X_train = X_train_non_clahe.astype('float32') / 255.0
X_val   = X_val_non_clahe.astype('float32') / 255.0
X_test  = X_test_non_clahe.astype('float32') / 255.0

print(f"Final shapes AFTER NORMALIZATION -> X_train_non_clahe: {X_train.shape}, X_val_non: {X_val.shape}, X_test_non: {X_test.shape}")


## ON-THE-FLY augmentation

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense, Flatten, Dropout, Input, LeakyReLU
)
from tensorflow.keras.callbacks import EarlyStopping

import tensorflow as tf
from tensorflow.keras import layers

import tensorflow as tf
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential(
    [
        # 1. FLIPS (Matches 'hflip' & 'vflip')
        # Your static code chooses one or the other; this enables both possibilities.
        layers.RandomFlip(mode="horizontal_and_vertical"),

        # 2. ROTATION (Matches 'rotate' 0-90 degrees)
        # Keras factor is a fraction of 2Pi. 90 deg / 360 deg = 0.25.
        # We use a tuple (0.0, 0.25) to ensure rotation is only positive (0 to 90), 
        # matching your static "np.random.uniform(0, 90)".
        layers.RandomRotation(
            factor=(0.0, 0.25), 
            fill_mode="reflect"
        ),

        # 3. SHIFT (Matches 'shift' ±5 pixels)
        # Image size is 224. 5 / 224 ≈ 0.0223.
        layers.RandomTranslation(
            height_factor=0.0223, 
            width_factor=0.0223, 
            fill_mode="reflect"
        ),

        # 4. SCALING (Matches 'scale' 0.9 to 1.1)
        # Your static code uses a scale matrix. 
        # 0.9 is Zoom Out, 1.1 is Zoom In.
        # Keras factor represents the percentage change: ±0.1.
        layers.RandomZoom(
            height_factor=(-0.1, 0.1), 
            width_factor=(-0.1, 0.1), 
            fill_mode="reflect"
        ),

        # 5. NOISE (Matches 'noise' Normal(0, 3))
        # Note: Keras models usually process inputs normalized to [0, 1].
        # Your static noise is std=3 on [0, 255] data.
        # Equivalent on [0, 1] data is 3/255 ≈ 0.0117.
        # If your model inputs are actually 0-255, change this back to 3.0.
        layers.GaussianNoise(stddev=3.0 / 255.0),
    ],
    name="data_augmentation",
)


#Datasets
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1024).batch(32)
val_ds   = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(32)
test_ds  = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(32)

In [4]:
# Hyperparameter options

activations = [
    ("relu", "relu"),
    ("leakyrelu", LeakyReLU(negative_slope=0.01))
]
batch_sizes =[32, 16, 8]


models_array=[]


for batch_size in batch_sizes:
    for activation_name, activation_fn in activations:
            print(
                f"Training model with: "
                f"activation={activation_name}, "
                f"batch_size={batch_size}"
            )
        
            #Apply early stopping to save time & avoid overfitting
            # Early stopping
            early_stop = EarlyStopping(patience=5, restore_best_weights=True,monitor='val_loss', verbose=1)


            # Model definition
            model = Sequential([
                Input(shape=(224, 224, 1)),
                data_augmentation,
                Conv2D(32, kernel_size=(3,3), activation=activation_fn),
                MaxPooling2D(pool_size=(2, 2)),

                Conv2D(32*2, kernel_size=(3,3), activation=activation_fn),
                MaxPooling2D(pool_size=(2, 2)),

                Conv2D(32*4, kernel_size=(3,3), activation=activation_fn),
                MaxPooling2D(pool_size=(2, 2)),

                Flatten(),
                Dense(128, activation=activation_fn),
                Dropout(0.5),
                Dense(1, activation='sigmoid')
            ])

            # Compile
            model.compile(
            optimizer='adam',
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

            # Train
            model.fit(
                X_train_non_clahe, y_train_non_clahe,
                validation_data=(X_val_non_clahe, y_val_non_clahe),
                epochs=30,
                batch_size=batch_size,
                callbacks=[early_stop]
            )

            models_array.append({
            "model": model,
            "activation": activation_name,
            "batch_size": batch_size
        })



Training model with activation=relu, conv_base_filter=32
Epoch 1/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 44s 596ms/step - accuracy: 0.6627 - loss: 0.8194 - val_accuracy: 0.5261 - val_loss: 0.6976
Epoch 2/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 45s 631ms/step - accuracy: 0.6307 - loss: 0.6398 - val_accuracy: 0.5130 - val_loss: 0.7225
Epoch 3/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 42s 587ms/step - accuracy: 0.6373 - loss: 0.6565 - val_accuracy: 0.5375 - val_loss: 0.6850
Epoch 4/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 43s 606ms/step - accuracy: 0.5836 - loss: 0.6747 - val_accuracy: 0.4951 - val_loss: 0.6937
Epoch 5/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 45s 632ms/step - accuracy: 0.5360 - loss: 0.6787 - val_accuracy: 0.5081 - val_loss: 0.6870
Epoch 6/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 44s 621ms/step - accuracy: 0.6400 - loss: 0.6611 - val_accuracy: 0.5293 - val_loss: 0.6877
Epoch 7/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 45s 637ms/step - accuracy: 0.5809 - loss: 0.6739 - val_accuracy: 0.5684 - val_loss: 0.6773
Epoch 8/30
71/71 ━━━━━━━━━━━━━━━━━━━━ 47

In [ ]:
from sklearn.metrics import fbeta_score

for model in models_array:
    print(f"{model['model']} ; {model['activation']} ; {model['batch_size']}")

    model = model['model']
    # Predict probabilities
    y_val_probs = model.predict(X_val_non_clahe)
    
    # Convert probabilities to class labels
    y_val_pred = (y_val_probs >= 0.5).astype(int).flatten()
    
    # Classification Report (includes precision, recall, f1-score per class)
    print("\nClassification Report:")
    print(model.name)
    print(classification_report(y_val_non_clahe, y_val_pred, digits=4))

    
    f2_score = fbeta_score(y_val_non_clahe, y_val_pred, beta=2, average='weighted')
    print("F2-score:", round(f2_score, 4))
 

    cm = confusion_matrix(y_val_non_clahe, y_val_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Calc', 'Mass'], yticklabels=['Calc', 'Mass'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix (Val Set)')
    plt.show()



# Explaining important conceps

### CNNs
- specialized NNs designed for grid-like data
- automatically learn spacial hierarchies of features through convolution operations
- highly effective in tasks involving visual perception

### Convolutional layer
**Convolution** is a mathematical operation where a small filter (kernel) is systematically applied across an input (such as an image) to produce a feature map that captures important local patterns like edges, textures, or more complex structures.

- **number of filters** = how many different kernels does the layer apply? So we basically get **number of filters** different feature maps.
- As we go on through the layers of the model, we may need more and more different kernels to try, because we have more intricate patterns in the images.

- **kernel_size** = what is the size of the filter/kernels, helping us to detect edges like shapes, edges etc.

- **activation** = the activation function used

### Pooling layers

A pooling layer reduces the spatial dimentions(width and height) of a feature map by summarizing regions of th input, helping to decrease computation, control overfitting, and make the network more robust to small translations

Purpose:

- downsampling: reduces the size of feature maps;
- feature preservation: keeps the most important information
- translation invariance: small changes in input data do not change the pooled output much

### Max Pooling

Helps us detect the strongest activation, e.g. what feature is the most present. This way, we can detect more proeminent patterns and also improve computations.

### Fully Connected Layers

Learn from the high-level features extracted by convolutional and pooling layers

- **dropout** - during training, turns off a percent of neurons in the Dense layer, to prevent overfitting.

### ReLU Activation
- keeps patterns of the data, also getting rid of negative values
- neurons stuck with negative inputs stop updating (gradient = 0)
- most popular in modern NNs

### LeakyReLU 
- allows a small gradient when x < 0 → avoids dying ReLU.

### Batch size
- how many training samples the network processes before updating its weights during training. It influences both the model's performance and the computational efficiency.
- some mostly used values are 32, 64, 128 ...
- I chose 32 because it puts in balance the computational resources needed and the efficiency of the model



### Adam Optimizer

Adam (short for **Adaptive Moment Estimation**) is one of the most popular and effective optimization algorithms used to train deep learning models. Adam adapts the learning rate for each parameter individually using **first** and **second moments** of the gradients:

- The **first moment** is the **mean** of the gradient (like momentum).
- The **second moment** is the **uncentered variance** of the gradient.

This helps Adam:
- Converge faster
- Handle sparse gradients
  